# QuantSafe additional experiments — resumable Google Drive edition
Run your original notebook **definition cells** first. Then run these extension cells in order. Every completed unit is atomically saved to Drive and skipped after Colab disconnects. The original notebook and outputs are not modified. Use CPU.

**Important:** Do not delete `QuantSafe_reviewer_extensions_v2/checkpoints`. Changing the configuration creates a separate experiment folder. E1/E2 are deterministic tests, while E3 is a paired stochastic comparison. A ZIP export cell is included at the end.

In [ ]:
# Run this cell AFTER the original notebook's controller/function definitions.
# Google Drive is the persistent checkpoint location. Do not use /content alone.
from google.colab import drive, files
from pathlib import Path
import os, json, hashlib, tempfile, zipfile, shutil, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import wilcoxon

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/QuantSafe_reviewer_extensions_v2')
RUNS = ROOT / 'checkpoints'
FINAL = ROOT / 'final_results'
for d in (RUNS, FINAL): d.mkdir(parents=True, exist_ok=True)
N_REPS = 5  # Set 100 for final E3 stochastic evaluation
N_STEPS = 5000
SEED_EXT = 120260
CONFIG = {'version':'reviewer_extension_v2_checkpointed','n_reps':N_REPS,'n_steps':N_STEPS,'seed':SEED_EXT,
          'experiments':['E1','E2','E3'], 'note':'E1/E2 are deterministic mechanism tests'}
CONFIG_ID = hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:12]
WORK = RUNS / CONFIG_ID
WORK.mkdir(parents=True, exist_ok=True)
(FINAL / f'config_{CONFIG_ID}.json').write_text(json.dumps(CONFIG,indent=2))
assert 'ImmediateExitHysteresisController' in globals(), 'Run original notebook definition cells first.'
assert all(x in globals() for x in ('bb84_key_fraction','ema_series','simulate_channel','evaluate_controller','Scenario'))

def atomic_json(path, obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    with open(tmp,'w') as f:
        json.dump(obj,f,allow_nan=False);f.flush();os.fsync(f.fileno())
    os.replace(tmp,path)

def save_once(name, make_record):
    # One checkpoint per experiment/rep/controller; skip completed tasks on restart.
    path=WORK/(name+'.json')
    if path.exists():
        try:
            old=json.loads(path.read_text())
            if old.get('complete') is True and old.get('config_id')==CONFIG_ID:
                return old['result'],True
        except (ValueError,KeyError,OSError): pass
    result=make_record()
    atomic_json(path,{'complete':True,'config_id':CONFIG_ID,'result':result})
    return result,False

def atomic_csv(df,path):
    path=Path(path);tmp=path.with_name(path.name+'.tmp')
    df.to_csv(tmp,index=False);os.replace(tmp,path)

print('Persistent checkpoints:',WORK)
print('Final outputs:',FINAL)
print('Configuration:',CONFIG_ID)
print('Changing N_REPS creates a NEW checkpoint namespace; completed 5-run tests are not silently mixed with 100-run results.')


## E1. Isolated emergency triggers
We use a fixed safe QBER of 0.04 for SKR-only and buffer-only tests. The SKR-only case deliberately overrides the BB84 proxy to model an *independent key-service outage*. This is a controlled counterfactual, not a physical QKD observation. The buffer-only case keeps the key fraction positive and forces low buffer occupancy. Each scenario is preceded by a stable recovery period to establish HYBRID operation. We record exit delay from the **first unsafe decision while HYBRID**, not from the start of an arbitrary attack interval.

In [ ]:
def isolated_trigger_trace(trigger, seed, length=300):
    rng=np.random.default_rng(seed)
    q=np.full(length,0.04)
    r=np.full(length,float(bb84_key_fraction(0.04)))
    buf=np.full(length,100.0)
    start=100
    if trigger=='qber_only':
        q[start:]=0.12  # key proxy deliberately remains positive to isolate QBER input
    elif trigger=='skr_only':
        r[start:]=0.0  # independent service outage, QBER remains safe
    elif trigger=='buffer_only':
        buf[start:]=24.0  # independent buffer depletion, QBER and key proxy safe
    else: raise ValueError(trigger)
    return pd.DataFrame({'qber_observed':q,'qber_ema':ema_series(q),'key_fraction':r,'buffer':buf}),start

def run_isolated(controller, inp, start):
    controller.reset(); modes=[]; reasons=[]
    for obs in inp.to_dict('records'):
        modes.append(controller.select(obs)); reasons.append(getattr(controller,'last_transition_reason',''))
    assert modes[start-1]=='HYBRID', 'Precondition failed: controller not in HYBRID'
    later=np.flatnonzero(np.array(modes[start:])=='PQC_ONLY')
    return {'exit_delay_decisions':int(later[0]) if len(later) else np.nan,
            'unsafe_hybrid_decisions':int(np.sum(np.array(modes[start:])=='HYBRID')),
            'first_exit_reason':reasons[start+int(later[0])] if len(later) else 'none'}

isolated=[]; reused=0
for trig in ['qber_only','skr_only','buffer_only']:
    for rep in range(N_REPS):
        def compute(trig=trig,rep=rep):
            inp,start=isolated_trigger_trace(trig,SEED_EXT+rep)
            return {'trigger':trig,'rep':rep,**run_isolated(ImmediateExitHysteresisController(),inp,start)}
        rec,skip=save_once(f'E1_{trig}_{rep:04d}',compute)
        isolated.append(rec);reused+=skip
isolated_df=pd.DataFrame(isolated)
atomic_csv(isolated_df,FINAL/f'E1_isolated_triggers_{CONFIG_ID}.csv')
print('E1 completed:',len(isolated),'reused:',reused)
print(isolated_df.groupby('trigger')[['exit_delay_decisions','unsafe_hybrid_decisions']].agg(['mean','max']))
print('Repeated deterministic scenarios are NOT independent stochastic evidence.')


## E2. Severe starvation with dynamic buffer accounting
A sustained, safe-QBER channel is used to isolate buffer gating. The key fraction remains positive, but generation is reduced to zero (or very low levels) after the controller enters HYBRID. We compare buffer-gated Immediate-Exit with a matched **no-buffer-gate ablation**. Both controllers use identical pre-decision generation and post-decision consumption; the no-gate version may be forced to PQC_ONLY by the simulator when fewer than one unit remains. Report separately (a) controller-gate exits and (b) simulator-enforced exhaustion. The initial buffer is deliberately small so depletion occurs within the test horizon; this is a stress test, not a calibrated field scenario.

In [ ]:
class NoBufferGate(ImmediateExitHysteresisController):
    name='Immediate-Exit without buffer gate'
    def select(self,obs):
        # Override ONLY the controller's buffer observation, not physical buffer accounting.
        x=dict(obs);x['buffer']=max(float(obs['buffer']),self.min_buffer)
        return super().select(x)

def starvation_trial(controller, seed, generation_per_step=0.0, initial=45., length=300):
    controller.reset();rng=np.random.default_rng(seed)
    buf=float(initial);q=0.04;r=float(bb84_key_fraction(q));qema=q
    rows=[];prev=None
    for t in range(length):
        # 0-29: safe recovery; 30 onward: sustained supply restriction
        generated=1.25 if t<30 else generation_per_step
        buf=min(BUFFER_MAX,buf+generated)
        obs={'qber_observed':q,'qber_ema':qema,'key_fraction':r,'buffer':buf}
        request=controller.select(obs);mode=request;enforced=False
        before=buf
        if mode=='HYBRID':
            if buf>=KEY_CONSUMPTION_PER_TX: buf-=KEY_CONSUMPTION_PER_TX
            else:
                enforced=True;mode='PQC_ONLY'
                controller.mode='PQC_ONLY';controller.recovery_count=0
        rows.append({'t':t,'buffer_before_decision':before,'buffer_after':buf,'requested_mode':request,
                     'mode':mode,'gate_exit':int(prev=='HYBRID' and mode=='PQC_ONLY' and not enforced and before<MIN_BUFFER_FOR_HYBRID),
                     'enforced_exhaustion':int(enforced),'hybrid':int(mode=='HYBRID'),
                     'below_reserve':int(before<MIN_BUFFER_FOR_HYBRID)})
        prev=mode
    z=pd.DataFrame(rows)
    return {'gate_exits':int(z.gate_exit.sum()),'enforced_exhaustions':int(z.enforced_exhaustion.sum()),
            'hybrid_fraction':z.hybrid.mean(),'min_buffer':z.buffer_after.min(),
            'below_reserve_decisions':int(z.below_reserve.sum())},z

starvation=[]; reused=0
for supply in [0.,0.1,0.5]:
    for rep in range(N_REPS):
        for label,cls in [('full',ImmediateExitHysteresisController),('no_buffer_gate',NoBufferGate)]:
            def compute(supply=supply,rep=rep,label=label,cls=cls):
                res,tr=starvation_trial(cls(),SEED_EXT+rep,generation_per_step=supply)
                if rep==0:
                    atomic_csv(tr,FINAL/f'E2_trace_supply_{supply}_{label}_{CONFIG_ID}.csv')
                return {'supply':supply,'rep':rep,'controller':label,**res}
            rec,skip=save_once(f'E2_supply_{supply}_{label}_{rep:04d}',compute)
            starvation.append(rec);reused+=skip
starvation_df=pd.DataFrame(starvation)
atomic_csv(starvation_df,FINAL/f'E2_starvation_summary_{CONFIG_ID}.csv')
print('E2 completed:',len(starvation),'reused:',reused)
print(starvation_df.groupby(['supply','controller'])[['gate_exits','enforced_exhaustions','hybrid_fraction','min_buffer']].mean())
print('E2 is a deterministic mechanism stress test, not a reliability estimate.')


## E3. Matched conventional hysteresis comparator
A conventional controller is defined with **raw QBER**, separate entry/exit thresholds, and no recovery dwell. Its safety exits for QBER >=0.10, key-unavailability, and low buffer are matched to Immediate-Exit; this avoids an unfair comparison with a controller that knowingly ignores emergency signals. The only policy difference is the recovery logic. This comparator is a transparent constructed baseline, **not** an implementation of a specific published controller. Evaluate on matched stochastic channel traces and report the operational trade-offs without claiming universal superiority.

In [ ]:
class ConventionalSafetyHysteresis(ImmediateExitHysteresisController):
    name='Conventional raw-QBER safety hysteresis'
    def select(self,obs):
        q=float(obs['qber_observed']);r=float(obs['key_fraction']);b=float(obs['buffer'])
        unsafe=q>=self.qkd_disable_qber or r<=0 or b<self.min_buffer
        if self.mode=='HYBRID':
            if unsafe or q>=self.theta_high:self.mode='PQC_ONLY'
        else:
            if q<=self.theta_low and r>0 and b>=self.min_buffer:self.mode='HYBRID'
        return self.mode

def matched_controller_trial(channel, controller, seed):
    # Uses the original evaluator's buffer and latency sampling rules.
    tr=evaluate_controller(channel,controller,seed=seed)
    return {'switches':int(tr.switch.sum()),'hybrid_fraction':float(tr.mode_selected.eq('HYBRID').mean()),
            'unsafe_hybrid_fraction':float(tr.unsafe_hybrid.mean()),
            'mean_latency_ms':float(tr.latency_ms.mean())}

comparison=[]; reused=0
for rep in range(N_REPS):
    seed=SEED_EXT+rep*101
    # The channel is generated only if at least one controller result is missing.
    tasks=[('Immediate-Exit',ImmediateExitHysteresisController),
           ('Conventional raw hysteresis',ConventionalSafetyHysteresis)]
    missing=[]
    for label,cls in tasks:
        key=f'E3_{rep:04d}_{"immediate" if label=="Immediate-Exit" else "conventional"}'
        path=WORK/(key+'.json')
        if not path.exists():missing.append((label,cls,key))
    channel=None
    if missing:
        channel=simulate_channel(n_tx=N_STEPS,seed=seed,scenario=Scenario(name='extension_baseline'))
    for label,cls in tasks:
        key=f'E3_{rep:04d}_{"immediate" if label=="Immediate-Exit" else "conventional"}'
        def compute(label=label,cls=cls,seed=seed):
            nonlocal_channel=channel
            if nonlocal_channel is None:
                nonlocal_channel=simulate_channel(n_tx=N_STEPS,seed=seed,scenario=Scenario(name='extension_baseline'))
            return {'rep':rep,'controller':label,**matched_controller_trial(nonlocal_channel,cls(),seed+9)}
        rec,skip=save_once(key,compute)
        comparison.append(rec);reused+=skip
    if (rep+1)%5==0 or rep+1==N_REPS:
        atomic_csv(pd.DataFrame(comparison),FINAL/f'E3_conventional_comparison_PARTIAL_{CONFIG_ID}.csv')
        print(f'E3 progress {rep+1}/{N_REPS} trajectories; reused {reused} checkpoints')
comparison_df=pd.DataFrame(comparison)
atomic_csv(comparison_df,FINAL/f'E3_conventional_comparison_{CONFIG_ID}.csv')
print(comparison_df.groupby('controller')[['switches','hybrid_fraction','unsafe_hybrid_fraction','mean_latency_ms']].agg(['mean','std']))
if N_REPS>=10:
    for metric in ['switches','hybrid_fraction','unsafe_hybrid_fraction','mean_latency_ms']:
        w=comparison_df.pivot(index='rep',columns='controller',values=metric)
        diff=w['Immediate-Exit']-w['Conventional raw hysteresis']
        if np.any(diff!=0):
            print(metric,'mean paired difference',diff.mean(),'Wilcoxon p (exploratory)',wilcoxon(diff).pvalue)
        else:print(metric,'identical on all paired runs')


## Interpretation and manuscript safeguards
- **E1**: correctness of isolated transitions; its repeated deterministic traces are not independent statistical trials.
- **E2**: mechanism activation and gate-versus-exhaustion distinction. Explicitly state that starting at 45 key units and imposing zero generation are synthetic stress assumptions.
- **E3**: compare recovery dwell/EMA versus a constructed safety-first hysteresis comparator. Not a new cryptographic primitive.
- **Telemetry**: the original simulator passes `key_fraction` derived from true QBER to the controller even when `qber_observed` is biased. That is a modeling assumption and may act as an independent safety signal; 
- **Latency**: sampled software proxy, not end-to-end RTGS performance.




In [ ]:
#e4
def e4_trial(rep,policy):
    seed=SEED+400000+rep*101
    sc=Scenario(name='e4_underreport',telemetry_bias=CONFIG['e4_bias'],attack_type='none')
    ch=simulate_channel(n_tx=N_STEPS,seed=seed,scenario=sc)
    # Bias is applied only during the specified attack window, not the whole trace.
    start=int(.30*N_STEPS);end=int(.60*N_STEPS)
    ch['qber_observed']=ch['qber_true'].copy()
    ch.loc[start:end-1,'qber_observed']=np.clip(ch.loc[start:end-1,'qber_true']+CONFIG['e4_bias'],.001,.499)
    rng=np.random.default_rng(seed+88)
    sensor=np.clip(ch.qber_true.to_numpy()+rng.normal(0,CONFIG['e4_sensor_sigma'],N_STEPS),.001,.499)
    disagreement=np.abs(ch.qber_observed.to_numpy()-sensor)>CONFIG['e4_disagreement_threshold']
    if policy=='consistency_fail_closed':
        ch['qber_observed']=np.where(disagreement,np.maximum(ch.qber_observed.to_numpy(),QKD_DISABLE_QBER),ch.qber_observed.to_numpy())
    tr=evaluate_controller(ch,ImmediateExitHysteresisController(),seed=seed+9)
    actual=tr.qber_true.to_numpy()>=QKD_DISABLE_QBER
    unsafe=np.logical_and(tr.mode_selected.eq('HYBRID').to_numpy(),actual)
    return {'rep':rep,'policy':policy,'true_channel_unsafe_decisions':int(unsafe.sum()),
            'true_channel_unsafe_fraction':float(unsafe.mean()),'hybrid_fraction':float(tr.mode_selected.eq('HYBRID').mean()),
            'switches':int(tr.switch.sum()),'mean_latency_ms':float(tr.latency_ms.mean()),
            'alarm_decisions':int(disagreement.sum()),'alarm_fraction':float(disagreement.mean()),
            'false_alarm_decisions':int(np.logical_and(disagreement,~actual).sum())}

rows=[]
for rep in range(N_E4):
    for policy in ['primary_only','consistency_fail_closed']:
        key=f'E4_{rep:04d}_{policy}'
        r,_=save_task(key,lambda rep=rep,policy=policy:e4_trial(rep,policy));rows.append(r)
    if (rep+1)%5==0:csv_atomic(pd.DataFrame(rows),OUT/f'E4_partial_{CONFIG_ID}.csv');print('E4',rep+1,'/',N_E4)
e4=pd.DataFrame(rows);csv_atomic(e4,OUT/f'E4_telemetry_{CONFIG_ID}.csv')
print(e4.groupby('policy')[['true_channel_unsafe_decisions','hybrid_fraction','false_alarm_decisions']].mean())

In [ ]:
P=np.asarray(P_EMP,float)
P_persist=.75*P+.25*np.eye(len(P));P_persist=P_persist/P_persist.sum(axis=1,keepdims=True)
STRESS={
 'baseline':dict(),
 'qber_shift_plus_0.01':dict(emission_shift=.01),
 'qber_scale_1.25':dict(emission_scale=1.25),
 'high_persistence':dict(transition_matrix=P_persist),
 'shift_and_persistence':dict(emission_shift=.01,transition_matrix=P_persist)
}
rows=[]
for rep in range(N_E5):
    seed=SEED+800000+rep*101
    for name,params in STRESS.items():
        key=f'E5_{rep:04d}_{name}'
        def run(rep=rep,seed=seed,name=name,params=params):
            ch=simulate_channel(n_tx=N_STEPS,seed=seed,scenario=Scenario(name=name,**params))
            tr=evaluate_controller(ch,ImmediateExitHysteresisController(),seed=seed+9)
            return {'rep':rep,'scenario':name,**metrics(tr),'mean_true_qber':float(ch.qber_true.mean())}
        r,_=save_task(key,run);rows.append(r)
    if (rep+1)%5==0:csv_atomic(pd.DataFrame(rows),OUT/f'E5_partial_{CONFIG_ID}.csv');print('E5',rep+1,'/',N_E5)
e5=pd.DataFrame(rows);csv_atomic(e5,OUT/f'E5_model_stress_{CONFIG_ID}.csv')
print(e5.groupby('scenario')[['switches','hybrid_fraction','unsafe_hybrid_fraction','mean_latency_ms','mean_true_qber']].mean())


In [ ]:
#Paired effect sizes and confidence intervals
def paired_ci(df,groupcol,a,b,metric,seed=91):
    w=df.pivot(index='rep',columns=groupcol,values=metric).dropna()
    d=(w[a]-w[b]).to_numpy();rng=np.random.default_rng(seed)
    means=np.mean(rng.choice(d,(2000,len(d)),replace=True),axis=1)
    p=float(wilcoxon(d).pvalue) if np.any(d!=0) else 1.
    return {'comparison':f'{a} minus {b}','metric':metric,'n':len(d),'mean_diff':float(d.mean()),
            'ci95_low':float(np.quantile(means,.025)),'ci95_high':float(np.quantile(means,.975)),
            'wilcoxon_p_exploratory':p}
summary=[]
for metric in ['switches','hybrid_fraction','mean_latency_ms','unsafe_hybrid_fraction']:
    summary.append(paired_ci(e3,'controller','Immediate-Exit','Conventional raw hysteresis',metric))
for metric in ['true_channel_unsafe_decisions','hybrid_fraction','false_alarm_decisions']:
    summary.append(paired_ci(e4,'policy','consistency_fail_closed','primary_only',metric))
for name in STRESS:
    if name=='baseline':continue
    for metric in ['switches','hybrid_fraction','mean_latency_ms']:
        summary.append(paired_ci(e5,'scenario',name,'baseline',metric))
summary_df=pd.DataFrame(summary)
csv_atomic(summary_df,OUT/f'paired_differences_{CONFIG_ID}.csv')
display(summary_df)


In [ ]:
expected=2*N_REPS+2*N_E4+len(STRESS)*N_E5
keys=[f'E3_{i:04d}_{c}' for i in range(N_REPS) for c in ['immediate','conventional']]+[f'E4_{i:04d}_{c}' for i in range(N_E4) for c in ['primary_only','consistency_fail_closed']]+[f'E5_{i:04d}_{s}' for i in range(N_E5) for s in STRESS]
missing=[k for k in keys if not (CK/(k+'.json')).exists()]
assert not missing,f'Missing {len(missing)} checkpoints: {missing[:5]}'
manifest={'config':CONFIG,'config_id':CONFIG_ID,'expected_tasks':expected,'task_count':len(keys),'source':'original QuantSafe notebook definitions; E3 old v2 checkpoints optional'}
(OUT/f'manifest_{CONFIG_ID}.json').write_text(json.dumps(manifest,indent=2))
zip_path=ROOT/f'QuantSafe_E3_E4_E5_{CONFIG_ID}.zip';tmp=zip_path.with_suffix('.zip.tmp')
with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for f in OUT.glob(f'*{CONFIG_ID}*'):z.write(f,'final_results/'+f.name)
    for k in keys:z.write(CK/(k+'.json'),'checkpoints/'+k+'.json')
    z.writestr('README.txt','E3 matched 100-run controller comparison; E4 simulated redundant QBER sensor; E5 synthetic model perturbations. Not a physical-QKD deployment or formal safety proof.\n')
os.replace(tmp,zip_path)
with zipfile.ZipFile(zip_path) as z:assert z.testzip() is None
print('Saved and verified:',zip_path, 'size MB:',round(zip_path.stat().st_size/1048576,2))
files.download(str(zip_path))



## Final export — ZIP and download
Run after E1–E3 finish. Includes final CSVs, configuration, all per-run JSON checkpoints, and a manifest. If you need to resume, reconnect Drive and rerun the notebook cells: completed units will be skipped. ZIP export itself never reruns experiments.

In [ ]:
# Create a self-contained ZIP on Drive, then download to your computer.
# This cell does not run any experiment.
#Final export — ZIP and download
# Create a self-contained ZIP on Drive, then download to your computer.
# This cell does not run any experiment.
required=[FINAL/f'E1_isolated_triggers_{CONFIG_ID}.csv',
          FINAL/f'E2_starvation_summary_{CONFIG_ID}.csv',
          FINAL/f'E3_conventional_comparison_{CONFIG_ID}.csv']
missing=[str(p) for p in required if not p.exists()]
if missing:
    raise RuntimeError('Cannot export a complete result bundle; missing: '+str(missing))
expected=3*N_REPS+6*N_REPS+2*N_REPS
checkpoints=list(WORK.glob('*.json'))
if len(checkpoints)!=expected:
    raise RuntimeError(f'Expected {expected} per-task checkpoints, found {len(checkpoints)}. Complete experiments first.')
manifest={'config':CONFIG,'config_id':CONFIG_ID,'expected_checkpoints':expected,
          'checkpoint_count':len(checkpoints),'note':'E1/E2 deterministic; E3 stochastic'}
atomic_json(FINAL/f'manifest_{CONFIG_ID}.json',manifest)
zip_path=ROOT/f'QuantSafe_reviewer_results_{CONFIG_ID}.zip'
tmp_zip=ROOT/f'.QuantSafe_reviewer_results_{CONFIG_ID}.zip.tmp'
with zipfile.ZipFile(tmp_zip,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for f in sorted(FINAL.glob(f'*{CONFIG_ID}*')):
        if f.is_file():z.write(f,arcname='final_results/'+f.name)
    for f in sorted(checkpoints):z.write(f,arcname='checkpoints/'+f.name)
    z.writestr('README.txt','QuantSafe reviewer experiments. Reproducibility requires original notebook definitions. Checkpoint JSONs are complete per-unit records. E1/E2 deterministic, E3 stochastic.\n')
os.replace(tmp_zip,zip_path)
with zipfile.ZipFile(zip_path) as z:
    bad=z.testzip()
    assert bad is None,f'ZIP validation failed: {bad}'
print('Verified ZIP saved permanently:',zip_path)
print('Size MB:',round(zip_path.stat().st_size/1024**2,2))
files.download(str(zip_path))
